# Build token-sequence features

In [ ]:
from __future__ import annotations

import ast
import json
import logging
import math
import os
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
PROJECT_ROOT = Path("/workspace")
INPUT_DIR = PROJECT_ROOT / "output" / "09_layer2_dataset"
OUTPUT_DIR = PROJECT_ROOT / "output" / "10_layer2_features"
LOG_DIR = PROJECT_ROOT / "output" / "logs"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

RUN_TAG = "09_build_token_features"
LOG_PATH = LOG_DIR / f"{RUN_TAG}.log"

MASTER_PATH = INPUT_DIR / "layer2_master_table.csv.gz"
TOKEN_EVENTS_PATH = INPUT_DIR / "layer2_token_events.csv.gz"
LEXICON_PATH = INPUT_DIR / "layer2_token_lexicon.csv"
BUILD_REPORT_PATH = INPUT_DIR / "layer2_build_report.json"
DIAG_PATH = INPUT_DIR / "layer2_diagnostics.json"

assert MASTER_PATH.exists(), f"Missing: {MASTER_PATH}"
assert TOKEN_EVENTS_PATH.exists(), f"Missing: {TOKEN_EVENTS_PATH}"
assert LEXICON_PATH.exists(), f"Missing: {LEXICON_PATH}"

In [ ]:
logger = logging.getLogger(RUN_TAG)
logger.setLevel(logging.INFO)
logger.handlers.clear()

fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")

fh = logging.FileHandler(LOG_PATH, mode="w")
fh.setLevel(logging.INFO)
fh.setFormatter(fmt)
logger.addHandler(fh)

sh = logging.StreamHandler()
sh.setLevel(logging.INFO)
sh.setFormatter(fmt)
logger.addHandler(sh)

logger.info("Starting Step 09 token feature build")
logger.info("INPUT_DIR=%s", INPUT_DIR)
logger.info("OUTPUT_DIR=%s", OUTPUT_DIR)

In [ ]:
def read_json(path: Path) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def parse_sequence_like(value):
    if value is None:
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, tuple):
        return list(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if pd.isna(value):
        return []
    s = str(value).strip()
    if not s:
        return []
    if s.startswith("[") and s.endswith("]"):
        try:
            obj = json.loads(s)
            if isinstance(obj, list):
                return obj
        except Exception:
            try:
                obj = ast.literal_eval(s)
                if isinstance(obj, list):
                    return obj
            except Exception:
                pass
    if "|" in s:
        return [x.strip() for x in s.split("|") if x.strip()]
    if "," in s:
        return [x.strip() for x in s.split(",") if x.strip()]
    return [s]

def normalize_unit_id(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return None
    s = str(x).strip()
    if s == "":
        return None
    s_low = s.lower()
    if s_low.startswith("u") and s_low[1:].isdigit():
        return f"u{int(s_low[1:]):02d}"
    if s_low.isdigit():
        return f"u{int(s_low):02d}"
    return s_low

def find_first_existing(df: pd.DataFrame, candidates: list[str], required: bool = True):
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise KeyError(f"None of these columns found: {candidates}")
    return None

def save_sparse_bundle(matrix, row_index: pd.DataFrame, feature_names: list[str], prefix: str):
    sparse.save_npz(OUTPUT_DIR / f"{prefix}.npz", matrix)
    row_index.to_csv(OUTPUT_DIR / f"{prefix}__rows.csv.gz", index=False, compression="gzip")
    pd.DataFrame({"feature": feature_names}).to_csv(
        OUTPUT_DIR / f"{prefix}__columns.csv.gz",
        index=False,
        compression="gzip"
    )

def entropy_from_counts(counts):
    counts = np.asarray(counts, dtype=float)
    total = counts.sum()
    if total <= 0:
        return 0.0
    p = counts[counts > 0] / total
    return float(-(p * np.log2(p)).sum())

def run_lengths(seq):
    if not seq:
        return []
    runs = []
    cur = seq[0]
    length = 1
    for tok in seq[1:]:
        if tok == cur:
            length += 1
        else:
            runs.append((cur, length))
            cur = tok
            length = 1
    runs.append((cur, length))
    return runs

def build_ngram_csr(sequences, vocabulary_tokens, n):
    vocab_set = set(vocabulary_tokens)
    feat_to_idx = {}
    rows, cols, vals = [], [], []
    for i, seq in enumerate(sequences):
        seq_f = [tok for tok in seq if tok in vocab_set]
        ctr = Counter(tuple(seq_f[j:j+n]) for j in range(len(seq_f) - n + 1))
        for ng, count in ctr.items():
            if ng not in feat_to_idx:
                feat_to_idx[ng] = len(feat_to_idx)
            rows.append(i)
            cols.append(feat_to_idx[ng])
            vals.append(float(count))
    X = sparse.csr_matrix((vals, (rows, cols)), shape=(len(sequences), len(feat_to_idx)), dtype=np.float32)
    feature_names = ["__".join(ng) for ng, _ in sorted(feat_to_idx.items(), key=lambda kv: kv[1])]
    return X, feature_names

def build_transition_csr(sequences, vocabulary_tokens, normalize=False):
    vocab = list(vocabulary_tokens)
    tok_to_idx = {tok: i for i, tok in enumerate(vocab)}
    n_vocab = len(vocab)
    rows, cols, vals = [], [], []

    for i, seq in enumerate(sequences):
        seq_f = [tok for tok in seq if tok in tok_to_idx]
        if len(seq_f) < 2:
            continue
        ctr = Counter((seq_f[j], seq_f[j+1]) for j in range(len(seq_f)-1))
        total_out = defaultdict(float)
        for (a, b), c in ctr.items():
            total_out[a] += c

        for (a, b), c in ctr.items():
            r = i
            cidx = tok_to_idx[a] * n_vocab + tok_to_idx[b]
            v = (c / total_out[a]) if normalize and total_out[a] > 0 else float(c)
            rows.append(r)
            cols.append(cidx)
            vals.append(float(v))

    X = sparse.csr_matrix((vals, (rows, cols)), shape=(len(sequences), n_vocab * n_vocab), dtype=np.float32)
    feature_names = [f"{a}=>{b}" for a in vocab for b in vocab]
    return X, feature_names

In [ ]:
master = pd.read_csv(MASTER_PATH, compression="gzip")
token_events = pd.read_csv(TOKEN_EVENTS_PATH, compression="gzip")
lexicon = pd.read_csv(LEXICON_PATH)

build_report = read_json(BUILD_REPORT_PATH) if BUILD_REPORT_PATH.exists() else {}
diag = read_json(DIAG_PATH) if DIAG_PATH.exists() else {}

logger.info("Loaded master rows=%s cols=%s", master.shape[0], master.shape[1])
logger.info("Loaded token_events rows=%s cols=%s", token_events.shape[0], token_events.shape[1])
logger.info("Loaded lexicon rows=%s cols=%s", lexicon.shape[0], lexicon.shape[1])

display(master.head(2))
display(token_events.head(2))
display(lexicon.head(2))

In [ ]:
record_key_col = find_first_existing(master, ["_record_key", "record_key", "clip_id"])
clip_id_col = find_first_existing(master, ["clip_id", "clip_name", "file_name"], required=False)

assert master[record_key_col].nunique(dropna=False) == len(master), (
    "Master table is not one-row-per-clip. Step 09 expects the Step 08 output."
)

logger.info("Clip-level grain validated: %s unique clips", len(master))

In [ ]:
sequence_col = find_first_existing(
    master,
    ["sequence_unit_ids", "sequence_tokens", "sequence", "tokens"],
    required=True
)

tokcount_cols = [c for c in master.columns if c.startswith("tokcount__")]
logger.info("Detected %s tokcount__ columns", len(tokcount_cols))

if not tokcount_cols:
    raise ValueError("No tokcount__* columns found. Step 08 output is required.")

In [ ]:
master = master.copy()
master["_sequence_tokens"] = master[sequence_col].apply(parse_sequence_like).apply(
    lambda seq: [normalize_unit_id(x) for x in seq if normalize_unit_id(x) is not None]
)

master["_sequence_len"] = master["_sequence_tokens"].apply(len)
logger.info(
    "Sequence lengths | min=%s | mean=%.3f | median=%.3f | max=%s",
    master["_sequence_len"].min(),
    master["_sequence_len"].mean(),
    master["_sequence_len"].median(),
    master["_sequence_len"].max(),
)

display(master[[record_key_col, sequence_col, "_sequence_tokens", "_sequence_len"]].head(3))

In [ ]:
lexicon = lexicon.copy()

if "unit_id" in lexicon.columns:
    lexicon["unit_id_norm"] = lexicon["unit_id"].apply(normalize_unit_id)
elif "token" in lexicon.columns:
    lexicon["unit_id_norm"] = lexicon["token"].apply(normalize_unit_id)
else:
    raise KeyError("Lexicon must contain either 'unit_id' or 'token'.")

lexicon["any_high_bias_flag"] = lexicon.get("any_high_bias_flag", False).fillna(False).astype(bool)
lexicon["any_bias_flag"] = lexicon.get("any_bias_flag", False).fillna(False).astype(bool)

full_vocab = sorted([x for x in lexicon["unit_id_norm"].dropna().unique().tolist()])
high_bias_filtered_vocab = sorted(
    lexicon.loc[~lexicon["any_high_bias_flag"], "unit_id_norm"].dropna().unique().tolist()
)

assert len(full_vocab) > 0, "Empty full vocabulary"
assert len(high_bias_filtered_vocab) > 0, "Empty high-bias-filtered vocabulary"

logger.info("Full vocab size=%s", len(full_vocab))
logger.info("High-bias-filtered vocab size=%s", len(high_bias_filtered_vocab))

pd.DataFrame({"unit_id": full_vocab}).to_csv(
    OUTPUT_DIR / "vocab_full.csv", index=False
)
lexicon.loc[lexicon["unit_id_norm"].isin(full_vocab)].to_csv(
    OUTPUT_DIR / "lexicon_full.csv.gz", index=False, compression="gzip"
)

pd.DataFrame({"unit_id": high_bias_filtered_vocab}).to_csv(
    OUTPUT_DIR / "vocab_high_bias_filtered.csv", index=False
)
lexicon.loc[lexicon["unit_id_norm"].isin(high_bias_filtered_vocab)].to_csv(
    OUTPUT_DIR / "lexicon_high_bias_filtered.csv.gz", index=False, compression="gzip"
)

display(lexicon[["unit_id_norm", "any_bias_flag", "any_high_bias_flag"]].head())

In [ ]:
index_cols_preferred = [
    record_key_col,
    clip_id_col if clip_id_col else None,
    "main_category_final",
    "sub_category_final",
    "farm",
    "barn_zone",
    "microphone",
    "mic_placement_context",
    "date",
    "day",
    "time",
]
index_cols = [c for c in index_cols_preferred if c and c in master.columns]

feature_index = master[index_cols].copy()
feature_index["row_id"] = np.arange(len(feature_index))

feature_index = feature_index[["row_id"] + [c for c in feature_index.columns if c != "row_id"]]
feature_index.to_csv(OUTPUT_DIR / "clip_feature_index.csv.gz", index=False, compression="gzip")
logger.info("Saved clip_feature_index with %s rows", len(feature_index))

display(feature_index.head())

In [ ]:
def build_unigram_tables(master_df, vocab_tokens):
    cols = [f"tokcount__{tok}" for tok in vocab_tokens if f"tokcount__{tok}" in master_df.columns]
    if not cols:
        raise ValueError("No unigram columns available for selected vocabulary.")
    X_counts = master_df[cols].fillna(0).astype(np.float32).copy()
    X_counts.columns = [c.replace("tokcount__", "") for c in X_counts.columns]
    row_sums = X_counts.sum(axis=1).replace(0, 1)
    X_props = X_counts.div(row_sums, axis=0).astype(np.float32)
    return X_counts, X_props

unigram_counts_full, unigram_props_full = build_unigram_tables(master, full_vocab)
unigram_counts_filt, unigram_props_filt = build_unigram_tables(master, high_bias_filtered_vocab)

unigram_counts_full.to_csv(OUTPUT_DIR / "unigram_counts_full.csv.gz", index=False, compression="gzip")
unigram_props_full.to_csv(OUTPUT_DIR / "unigram_props_full.csv.gz", index=False, compression="gzip")
unigram_counts_filt.to_csv(OUTPUT_DIR / "unigram_counts_high_bias_filtered.csv.gz", index=False, compression="gzip")
unigram_props_filt.to_csv(OUTPUT_DIR / "unigram_props_high_bias_filtered.csv.gz", index=False, compression="gzip")

logger.info("Saved unigram count/prop tables")

In [ ]:
te = token_events.copy()

te_record_key_col = find_first_existing(te, [record_key_col, "_record_key", "record_key", "clip_id"])
te_unit_col = find_first_existing(te, ["unit_id", "token", "unit", "cluster_id"])
te_unit_col_norm = "_unit_id_norm"
te[te_unit_col_norm] = te[te_unit_col].apply(normalize_unit_id)

duration_col = find_first_existing(
    te,
    ["segment_duration_sec", "duration_sec", "dur_sec", "segment_len_sec", "duration"],
    required=False
)

if duration_col is None:
    logger.warning("No duration column found in token_events. Falling back to weight=1.0 per event.")
    te["_event_weight"] = 1.0
else:
    te["_event_weight"] = pd.to_numeric(te[duration_col], errors="coerce").fillna(0.0).astype(np.float32)

dur_long = (
    te.groupby([te_record_key_col, te_unit_col_norm], dropna=False)["_event_weight"]
      .sum()
      .reset_index()
      .rename(columns={te_record_key_col: record_key_col, te_unit_col_norm: "unit_id", "_event_weight": "duration_weight"})
)

dur_wide = (
    dur_long.pivot(index=record_key_col, columns="unit_id", values="duration_weight")
            .fillna(0.0)
            .astype(np.float32)
)

dur_wide = dur_wide.reindex(master[record_key_col]).fillna(0.0)
dur_wide.index = master.index

def duration_tables_for_vocab(vocab_tokens, suffix):
    cols = [tok for tok in vocab_tokens if tok in dur_wide.columns]
    X_dur = dur_wide[cols].copy()
    X_dur.columns = [f"durw__{c}" for c in X_dur.columns]
    row_sums = X_dur.sum(axis=1).replace(0, 1)
    X_dur_prop = X_dur.div(row_sums, axis=0).astype(np.float32)
    X_dur.to_csv(OUTPUT_DIR / f"duration_weighted_{suffix}.csv.gz", index=False, compression="gzip")
    X_dur_prop.to_csv(OUTPUT_DIR / f"duration_weighted_prop_{suffix}.csv.gz", index=False, compression="gzip")
    return X_dur, X_dur_prop

duration_full, duration_prop_full = duration_tables_for_vocab(full_vocab, "full")
duration_filt, duration_prop_filt = duration_tables_for_vocab(high_bias_filtered_vocab, "high_bias_filtered")

logger.info("Saved duration-weighted unigram features")

In [ ]:
def summarize_sequence(seq):
    seq = [x for x in seq if x is not None]
    counts = Counter(seq)
    runs = run_lengths(seq)
    transitions = len(seq) - 1 if len(seq) >= 2 else 0

    same_adj = 0
    for a, b in zip(seq[:-1], seq[1:]):
        if a == b:
            same_adj += 1

    run_lengths_only = [rl for _, rl in runs]
    top_count = max(counts.values()) if counts else 0
    entropy = entropy_from_counts(list(counts.values()))
    max_entropy = math.log2(len(counts)) if len(counts) > 1 else 0.0
    norm_entropy = (entropy / max_entropy) if max_entropy > 0 else 0.0

    return {
        "seq_len": len(seq),
        "n_unique_tokens": len(counts),
        "type_token_ratio": (len(counts) / len(seq)) if len(seq) > 0 else 0.0,
        "sequence_entropy": entropy,
        "sequence_entropy_norm": norm_entropy,
        "n_runs": len(runs),
        "mean_run_length": float(np.mean(run_lengths_only)) if run_lengths_only else 0.0,
        "max_run_length": max(run_lengths_only) if run_lengths_only else 0,
        "singleton_run_fraction": (sum(1 for x in run_lengths_only if x == 1) / len(run_lengths_only)) if run_lengths_only else 0.0,
        "transition_count": transitions,
        "repeat_adjacent_fraction": (same_adj / transitions) if transitions > 0 else 0.0,
        "dominant_token_fraction": (top_count / len(seq)) if len(seq) > 0 else 0.0,
        "first_token": seq[0] if seq else None,
        "last_token": seq[-1] if seq else None,
    }

seq_summary = master["_sequence_tokens"].apply(summarize_sequence).apply(pd.Series)

for c in ["n_segments", "clip_duration_sec", "_raw_sequence_len"]:
    if c in master.columns and c not in seq_summary.columns:
        seq_summary[c] = pd.to_numeric(master[c], errors="coerce")

for which in ["first_token", "last_token"]:
    if which in seq_summary.columns:
        dummies = pd.get_dummies(seq_summary[which], prefix=which, dtype=np.uint8)
        seq_summary = pd.concat([seq_summary.drop(columns=[which]), dummies], axis=1)

seq_summary.to_csv(OUTPUT_DIR / "sequence_summary_features.csv.gz", index=False, compression="gzip")
logger.info("Saved sequence-summary features with shape=%s", seq_summary.shape)

display(seq_summary.head(3))

In [ ]:
sequences_all = master["_sequence_tokens"].tolist()
row_index = feature_index.copy()

def sequence_docs(sequences, allowed_vocab):
    allowed = set(allowed_vocab)
    docs = []
    for seq in sequences:
        toks = [tok for tok in seq if tok in allowed]
        docs.append(" ".join(toks))
    return docs

X_bigram_full, feat_bigram_full = build_ngram_csr(sequences_all, full_vocab, n=2)
X_trigram_full, feat_trigram_full = build_ngram_csr(sequences_all, full_vocab, n=3)
X_trans_count_full, feat_trans_full = build_transition_csr(sequences_all, full_vocab, normalize=False)
X_trans_prob_full, _ = build_transition_csr(sequences_all, full_vocab, normalize=True)

save_sparse_bundle(X_bigram_full, row_index, feat_bigram_full, "bigram_counts_full")
save_sparse_bundle(X_trigram_full, row_index, feat_trigram_full, "trigram_counts_full")
save_sparse_bundle(X_trans_count_full, row_index, feat_trans_full, "transition_counts_full")
save_sparse_bundle(X_trans_prob_full, row_index, feat_trans_full, "transition_probs_full")

docs_full = sequence_docs(sequences_all, full_vocab)
tfidf_full = TfidfVectorizer(
    analyzer="word",
    lowercase=False,
    token_pattern=r"(?u)\b\w+\b",
    ngram_range=(1, 3),
    min_df=1,
    dtype=np.float32,
)
X_tfidf_full = tfidf_full.fit_transform(docs_full)
feat_tfidf_full = tfidf_full.get_feature_names_out().tolist()
save_sparse_bundle(X_tfidf_full, row_index, feat_tfidf_full, "tfidf_1to3_full")

X_bigram_filt, feat_bigram_filt = build_ngram_csr(sequences_all, high_bias_filtered_vocab, n=2)
X_trigram_filt, feat_trigram_filt = build_ngram_csr(sequences_all, high_bias_filtered_vocab, n=3)
X_trans_count_filt, feat_trans_filt = build_transition_csr(sequences_all, high_bias_filtered_vocab, normalize=False)
X_trans_prob_filt, _ = build_transition_csr(sequences_all, high_bias_filtered_vocab, normalize=True)

save_sparse_bundle(X_bigram_filt, row_index, feat_bigram_filt, "bigram_counts_high_bias_filtered")
save_sparse_bundle(X_trigram_filt, row_index, feat_trigram_filt, "trigram_counts_high_bias_filtered")
save_sparse_bundle(X_trans_count_filt, row_index, feat_trans_filt, "transition_counts_high_bias_filtered")
save_sparse_bundle(X_trans_prob_filt, row_index, feat_trans_filt, "transition_probs_high_bias_filtered")

docs_filt = sequence_docs(sequences_all, high_bias_filtered_vocab)
tfidf_filt = TfidfVectorizer(
    analyzer="word",
    lowercase=False,
    token_pattern=r"(?u)\b\w+\b",
    ngram_range=(1, 3),
    min_df=1,
    dtype=np.float32,
)
X_tfidf_filt = tfidf_filt.fit_transform(docs_filt)
feat_tfidf_filt = tfidf_filt.get_feature_names_out().tolist()
save_sparse_bundle(X_tfidf_filt, row_index, feat_tfidf_filt, "tfidf_1to3_high_bias_filtered")

logger.info("Saved sparse n-gram, transition, and TF-IDF bundles")
logger.info("bigram_full shape=%s", X_bigram_full.shape)
logger.info("trigram_full shape=%s", X_trigram_full.shape)
logger.info("tfidf_full shape=%s", X_tfidf_full.shape)
logger.info("transition_full shape=%s", X_trans_count_full.shape)

In [ ]:
def zero_row_fraction_from_df(df):
    arr = df.to_numpy(dtype=np.float32)
    return float((arr.sum(axis=1) == 0).mean())

def zero_row_fraction_from_sparse(X):
    return float((np.asarray(X.sum(axis=1)).ravel() == 0).mean())

feature_manifest = {
    "input_dir": str(INPUT_DIR),
    "output_dir": str(OUTPUT_DIR),
    "n_clips": int(len(master)),
    "record_key_col": record_key_col,
    "sequence_col": sequence_col,
    "token_events_rows": int(len(token_events)),
    "full_vocab_size": int(len(full_vocab)),
    "high_bias_filtered_vocab_size": int(len(high_bias_filtered_vocab)),
    "bias_filter_rule": "exclude any_high_bias_flag == True",
    "outputs": {
        "clip_feature_index": "clip_feature_index.csv.gz",
        "unigram_counts_full": "unigram_counts_full.csv.gz",
        "unigram_props_full": "unigram_props_full.csv.gz",
        "unigram_counts_high_bias_filtered": "unigram_counts_high_bias_filtered.csv.gz",
        "unigram_props_high_bias_filtered": "unigram_props_high_bias_filtered.csv.gz",
        "duration_weighted_full": "duration_weighted_full.csv.gz",
        "duration_weighted_prop_full": "duration_weighted_prop_full.csv.gz",
        "duration_weighted_high_bias_filtered": "duration_weighted_high_bias_filtered.csv.gz",
        "duration_weighted_prop_high_bias_filtered": "duration_weighted_prop_high_bias_filtered.csv.gz",
        "sequence_summary_features": "sequence_summary_features.csv.gz",
        "bigram_counts_full": "bigram_counts_full.npz",
        "trigram_counts_full": "trigram_counts_full.npz",
        "tfidf_1to3_full": "tfidf_1to3_full.npz",
        "transition_counts_full": "transition_counts_full.npz",
        "transition_probs_full": "transition_probs_full.npz",
        "bigram_counts_high_bias_filtered": "bigram_counts_high_bias_filtered.npz",
        "trigram_counts_high_bias_filtered": "trigram_counts_high_bias_filtered.npz",
        "tfidf_1to3_high_bias_filtered": "tfidf_1to3_high_bias_filtered.npz",
        "transition_counts_high_bias_filtered": "transition_counts_high_bias_filtered.npz",
        "transition_probs_high_bias_filtered": "transition_probs_high_bias_filtered.npz",
    },
}

diagnostics = {
    "n_clips": int(len(master)),
    "full_vocab_size": int(len(full_vocab)),
    "high_bias_filtered_vocab_size": int(len(high_bias_filtered_vocab)),
    "sequence_length": {
        "min": float(master["_sequence_len"].min()),
        "mean": float(master["_sequence_len"].mean()),
        "median": float(master["_sequence_len"].median()),
        "max": float(master["_sequence_len"].max()),
    },
    "dense_shapes": {
        "unigram_counts_full": list(unigram_counts_full.shape),
        "unigram_props_full": list(unigram_props_full.shape),
        "unigram_counts_high_bias_filtered": list(unigram_counts_filt.shape),
        "unigram_props_high_bias_filtered": list(unigram_props_filt.shape),
        "duration_weighted_full": list(duration_full.shape),
        "duration_weighted_high_bias_filtered": list(duration_filt.shape),
        "sequence_summary_features": list(seq_summary.shape),
    },
    "sparse_shapes": {
        "bigram_counts_full": list(X_bigram_full.shape),
        "trigram_counts_full": list(X_trigram_full.shape),
        "tfidf_1to3_full": list(X_tfidf_full.shape),
        "transition_counts_full": list(X_trans_count_full.shape),
        "bigram_counts_high_bias_filtered": list(X_bigram_filt.shape),
        "trigram_counts_high_bias_filtered": list(X_trigram_filt.shape),
        "tfidf_1to3_high_bias_filtered": list(X_tfidf_filt.shape),
        "transition_counts_high_bias_filtered": list(X_trans_count_filt.shape),
    },
    "zero_row_fraction": {
        "unigram_counts_full": zero_row_fraction_from_df(unigram_counts_full),
        "unigram_counts_high_bias_filtered": zero_row_fraction_from_df(unigram_counts_filt),
        "duration_weighted_full": zero_row_fraction_from_df(duration_full),
        "duration_weighted_high_bias_filtered": zero_row_fraction_from_df(duration_filt),
        "bigram_counts_full": zero_row_fraction_from_sparse(X_bigram_full),
        "trigram_counts_full": zero_row_fraction_from_sparse(X_trigram_full),
        "tfidf_1to3_full": zero_row_fraction_from_sparse(X_tfidf_full),
        "transition_counts_full": zero_row_fraction_from_sparse(X_trans_count_full),
        "bigram_counts_high_bias_filtered": zero_row_fraction_from_sparse(X_bigram_filt),
        "trigram_counts_high_bias_filtered": zero_row_fraction_from_sparse(X_trigram_filt),
        "tfidf_1to3_high_bias_filtered": zero_row_fraction_from_sparse(X_tfidf_filt),
        "transition_counts_high_bias_filtered": zero_row_fraction_from_sparse(X_trans_count_filt),
    },
}

with open(OUTPUT_DIR / "feature_manifest.json", "w", encoding="utf-8") as f:
    json.dump(feature_manifest, f, indent=2)

with open(OUTPUT_DIR / "feature_diagnostics.json", "w", encoding="utf-8") as f:
    json.dump(diagnostics, f, indent=2)

logger.info("Saved feature manifest and diagnostics")
display(pd.json_normalize(diagnostics, sep=" | "))